### SECTION01. 로지스틱 회귀 분석

In [1]:
import numpy as np
import pandas as pd
from statsmodels.formula.api import logit

In [6]:
df = pd.read_csv('health_survey2.csv')
df.head()

,age,bmi,smoker,activity_level,disease,lifestyle
0,62,35.2,0,0,1,S
1,65,18.6,0,2,1,A
2,71,33.2,0,1,1,M
3,18,37.1,1,2,0,A
4,21,17.6,0,0,0,S


### 1.통계요약

In [8]:
model = logit('disease~age+bmi+smoker+activity_level+lifestyle',data=df).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.637083
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      993
Method:                           MLE   Df Model:                            6
Date:                Fri, 13 Nov 2026   Pseudo R-squ.:                 0.05977
Time:                        11:33:18   Log-Likelihood:                -637.08
converged:                       True   LL-Null:                       -677.58
Covariance Type:            nonrobust   LLR p-value:                 2.230e-15
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.7060      0.321     -5.320      0.000      -2.335      -1.078
lifestyle[T.M]    -0.0631      0.170     -0.370      0.711      -0.397       0.271
lifestyle[T.S]    -0.0793      0.177     -0.447      0.655      -0.427       0.268
age                0.0184      0.004      4.815      0.000       0.011       0.026
bmi                0.0566      0.009      6.403      0.000       0.039       0.074
smoker             0.3127      0.149      2.095      0.036       0.020       0.605
activity_level    -0.2507      0.086     -2.924      0.003      -0.419      -0.083
==================================================================================
"""

### 2.bmi변수의 계수값

In [14]:
print("bmi계수 :",model.params['bmi'])

bmi계수 : 0.05658114050609094


### 3.bmi 변수가 한 단위 증가할 때와 세 단위 증가할 때의 오즈비 값은?

In [16]:
unit1 = np.exp(model.params['bmi'])
unit3 = np.exp(model.params['bmi']*3)

print("한 단위 :", unit1)
print("세 단위 :", unit3)

한 단위 : 1.0582124752125908
세 단위 : 1.1850007666136944


### 4.생활스타일 간의 오즈비

In [19]:
print("A대비 M의 오즈비 :", np.exp(model.params['lifestyle[T.M]']))
print("A대비 S의 오즈비 :", np.exp(model.params['lifestyle[T.S]']))

A대비 M의 오즈비 : 0.9388330987984917
A대비 S의 오즈비 : 0.9238046412095142


In [20]:
print("M대비 A의 오즈비 :", np.exp(-model.params['lifestyle[T.M]']))
print("S대비 A의 오즈비 :", np.exp(-model.params['lifestyle[T.S]']))

M대비 A의 오즈비 : 1.0651520502204161
S대비 A의 오즈비 : 1.0824799480231286


In [21]:
print("M대비 S의 오즈비 :", np.exp(model.params['lifestyle[T.S]']-model.params['lifestyle[T.M]']))
print("S대비 M의 오즈비 :", np.exp(-model.params['lifestyle[T.M]']-model.params['lifestyle[T.S]']))

M대비 S의 오즈비 : 0.9839924075874501
S대비 M의 오즈비 : 1.153005735959325


### 로지스틱 회귀의 예측

주어진 데이터를 train 700개, test 300개로 분할하시오. age,bmi,smoker,lifestyle을 독립변수로 사용하고 disease를 종속 변수로 하는 로지스틱 회귀모델을 train 데이터로 적합시키시오. 적합된 모델로 test 데이터를 예측했을 때, 질병발생확률이 가장 높은 사람의 bmi값을 구하시오.

In [23]:
train = df.iloc[:700]
test = df.iloc[700:]

print("train :", train.shape)
print("test :", test.shape)

train : (700, 6)
test : (300, 6)


In [24]:
model = logit('disease~age+bmi+smoker+lifestyle',data=train).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.637966
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                disease   No. Observations:                  700
Model:                          Logit   Df Residuals:                      694
Method:                           MLE   Df Model:                            5
Date:                Fri, 13 Nov 2026   Pseudo R-squ.:                 0.05746
Time:                        14:25:37   Log-Likelihood:                -446.58
converged:                       True   LL-Null:                       -473.80
Covariance Type:            nonrobust   LLR p-value:                 1.694e-10
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0554      0.368     -5.579      0.000      -2.778      -1.333
lifestyle[T.M]     0.0368      0.198      0.186      0.852      -0.351       0.425
lifestyle[T.S]     0.3639      0.201      1.808      0.071      -0.031       0.758
age                0.0185      0.005      4.083      0.000       0.010       0.027
bmi                0.0544      0.010      5.182      0.000       0.034       0.075
smoker             0.2650      0.178      1.485      0.138      -0.085       0.615
==================================================================================
"""

In [26]:
pred = model.predict(test)
idx = pred.idxmax()

In [29]:
print("확률이 가장 큰 사람의 bmi :", test.loc[idx, 'bmi'])

확률이 가장 큰 사람의 bmi : 42.2


In [35]:
for idx,val in model.pvalues.items():
    if val < 0.05:
        print(idx)

Intercept
age
bmi


In [36]:
model = logit('disease~age+bmi', data=train).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.642623
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                disease   No. Observations:                  700
Model:                          Logit   Df Residuals:                      697
Method:                           MLE   Df Model:                            2
Date:                Fri, 13 Nov 2026   Pseudo R-squ.:                 0.05058
Time:                        14:34:38   Log-Likelihood:                -449.84
converged:                       True   LL-Null:                       -473.80
Covariance Type:            nonrobust   LLR p-value:                 3.909e-11
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.8710      0.343     -5.455      0.000      -2.543      -1.199
age            0.0197      0.004      4.395      0.000       0.011       0.028
bmi            0.0535      0.010      5.129      0.000       0.033       0.074
==============================================================================
"""

In [40]:
pred = model.predict(test)
pred = (pred > 0.4).astype(int)

In [41]:
from sklearn.metrics import recall_score
print("민감도 :", recall_score(test['disease'], pred))

민감도 : 0.96
